# NOTEBOOK 1 - Data Preparation & Augmentation (`tongue-dataprep`)

This notebook:
1. Loads `t2d_tongue_dataset` from Kaggle Input.
2. Prints class distribution before augmentation.
3. Applies **offline** Albumentations augmentation per class (with controlled oversampling for balance).
4. Splits the **fully augmented** dataset into `70/15/15` (`train/val/test`) with stratification.
5. Prevents leakage by keeping each original image and all its augmentations in the same split.
6. Saves final folders directly to `/kaggle/working/{train,val,test}/class_name/`.

Publish `/kaggle/working` as a Kaggle Dataset named `tongue_preped`.

In [1]:
# If needed on Kaggle, uncomment:
# !pip install -q albumentations==1.4.18 opencv-python-headless==4.10.0.84

import os
import re
import cv2
import shutil
import random
import hashlib
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import albumentations as A
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

INPUT_BASE = Path('/kaggle/input')
# Primary path you provided from training notebook:
DATA_ROOT = Path('/kaggle/input/datasets/edamhakim/t2d-tongue-dataset/data/preprocessedcropped')

OUTPUT_BASE = Path('/kaggle/working')
SPLITS = ('train', 'val', 'test')
SPLIT_RATIOS = (0.70, 0.15, 0.15)
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

CONFIG = {
    'seed': SEED,
    'base_aug_per_image': 1,
    'max_aug_per_family': 6,
    'save_jpg_quality': 95,
    'run_hash_leakage_check': True,
}

assert sum(SPLIT_RATIOS) == 1.0, 'Split ratios must sum to 1.0'

In [2]:
def is_image_file(path: Path) -> bool:
    return path.suffix.lower() in IMAGE_EXTS


def canonical_key(path: Path) -> str:
    """Build a grouping key so source pre-augmented variants map to one family."""
    stem = path.stem.lower()
    stem = re.sub(r'(_?flip|_?rot(?:ate)?(?:_neg)?_?\d*|_?resized|_?aug(?:_\d+)?)', '', stem)
    stem = re.sub(r'__+', '_', stem).strip('_')
    return stem


def collect_originals(root: Path):
    class_to_files = defaultdict(list)
    class_to_groups = defaultdict(lambda: defaultdict(list))

    # Support either root/class/*.jpg OR deeper layouts (train/val/test/class/*.jpg)
    for img_path in root.rglob('*'):
        if img_path.is_file() and is_image_file(img_path):
            class_name = img_path.parent.name.strip().lower()
            class_to_files[class_name].append(img_path)
            class_to_groups[class_name][canonical_key(img_path)].append(img_path)

    class_to_files = {k: sorted(v) for k, v in class_to_files.items() if len(v) > 0}

    if len(class_to_files) < 2:
        raise ValueError(
            f'Expected at least 2 classes, found {len(class_to_files)}. '
            'Check your dataset folder layout.'
        )

    # Pick one representative path per canonical group (prefer non-aug-looking names)
    class_to_representatives = {}
    for cls, groups in class_to_groups.items():
        reps = []
        for _, paths in groups.items():
            paths = sorted(paths)
            preferred = [p for p in paths if 'aug' not in p.stem.lower() and 'flip' not in p.stem.lower() and 'rot' not in p.stem.lower()]
            reps.append(preferred[0] if preferred else paths[0])
        class_to_representatives[cls] = sorted(reps)

    return class_to_files, class_to_representatives


def print_distribution(title: str, counts: dict):
    print(f'\n{title}')
    df = pd.DataFrame({'class': list(counts.keys()), 'count': list(counts.values())})
    print(df.sort_values('class').to_string(index=False))


def md5_file(path: Path) -> str:
    h = hashlib.md5()
    with open(path, 'rb') as f:
        while True:
            chunk = f.read(1024 * 1024)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

In [3]:
if not DATA_ROOT.exists():
    raise FileNotFoundError(f'DATA_ROOT not found: {DATA_ROOT}')

dataset_root = DATA_ROOT
print(f'Dataset root: {dataset_root}')

class_to_all_files, class_to_originals = collect_originals(dataset_root)

all_counts = {cls: len(paths) for cls, paths in class_to_all_files.items()}
print_distribution('Class distribution FOUND in source path:', all_counts)

original_counts = {cls: len(paths) for cls, paths in class_to_originals.items()}
print_distribution('Class distribution BEFORE augmentation (deduplicated originals):', original_counts)

Dataset root: /kaggle/input/datasets/edamhakim/t2d-tongue-dataset/data/preprocessedcropped

Class distribution FOUND in source path:
      class  count
   diabetes    250
nondiabetes    250

Class distribution BEFORE augmentation (deduplicated originals):
      class  count
   diabetes    249
nondiabetes    250


In [4]:
# Albumentations pipeline (prefer Albumentations over custom code)
augmenter = A.Compose([
    A.OneOf([
        A.HorizontalFlip(p=1.0),
        A.VerticalFlip(p=1.0),
        A.RandomRotate90(p=1.0),
    ], p=0.6),
    A.ShiftScaleRotate(
        shift_limit=0.05,
        scale_limit=0.10,
        rotate_limit=25,
        border_mode=cv2.BORDER_REFLECT_101,
        p=0.8,
    ),
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1.0),
        A.CLAHE(clip_limit=2.5, tile_grid_size=(8, 8), p=1.0),
        A.HueSaturationValue(hue_shift_limit=8, sat_shift_limit=15, val_shift_limit=10, p=1.0),
    ], p=0.7),
    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 5), p=1.0),
        A.GaussNoise(std_range=(0.03, 0.08), p=1.0),
        A.MotionBlur(blur_limit=5, p=1.0),
    ], p=0.35),
], p=1.0)


def augment_once(img_bgr: np.ndarray) -> np.ndarray:
    return augmenter(image=img_bgr)['image']


# Build family records: each representative original image = one family
# We keep each family in exactly one split to prevent leakage.
families = []
for cls, paths in class_to_originals.items():
    for i, p in enumerate(paths):
        family_id = f'{cls}_{i:06d}_{canonical_key(p)}'
        families.append({
            'family_id': family_id,
            'group_id': canonical_key(p),
            'class_name': cls,
            'src_path': p,
        })

# Augmentation policy:
# - At least CONFIG['base_aug_per_image'] augmentations per original for all classes
# - Then controlled oversampling to balance classes, with max cap per family
base_aug_per_image = CONFIG['base_aug_per_image']
base_counts = {
    cls: len(paths) * (1 + base_aug_per_image)
    for cls, paths in class_to_originals.items()
}
target_count = max(base_counts.values())

family_aug_counts = {f['family_id']: base_aug_per_image for f in families}

for cls, paths in class_to_originals.items():
    current = base_counts[cls]
    needed = target_count - current
    if needed <= 0:
        continue

    class_family_ids = [f['family_id'] for f in families if f['class_name'] == cls]
    idx = 0
    while needed > 0:
        fid = class_family_ids[idx % len(class_family_ids)]
        if family_aug_counts[fid] < CONFIG['max_aug_per_family']:
            family_aug_counts[fid] += 1
            needed -= 1
        idx += 1
        # safety break when cap blocks further balancing
        if idx > len(class_family_ids) * (CONFIG['max_aug_per_family'] + 2):
            break

augmented_counts = defaultdict(int)
for f in families:
    cls = f['class_name']
    fid = f['family_id']
    augmented_counts[cls] += 1 + family_aug_counts[fid]

print_distribution('Class distribution AFTER augmentation (planned):', dict(augmented_counts))


# Stratified split by family labels (70/15/15)
family_ids = [f['family_id'] for f in families]
labels = [f['class_name'] for f in families]

train_ids, temp_ids, train_y, temp_y = train_test_split(
    family_ids,
    labels,
    test_size=(1 - SPLIT_RATIOS[0]),
    random_state=SEED,
    stratify=labels,
)

val_ratio_within_temp = SPLIT_RATIOS[1] / (SPLIT_RATIOS[1] + SPLIT_RATIOS[2])
val_ids, test_ids, _, _ = train_test_split(
    temp_ids,
    temp_y,
    test_size=(1 - val_ratio_within_temp),
    random_state=SEED,
    stratify=temp_y,
)

split_map = {}
for fid in train_ids:
    split_map[fid] = 'train'
for fid in val_ids:
    split_map[fid] = 'val'
for fid in test_ids:
    split_map[fid] = 'test'

assert len(split_map) == len(family_ids)

# Ensure group leakage is impossible at plan time
split_to_groups = defaultdict(set)
for f in families:
    split_to_groups[split_map[f['family_id']]].add(f['group_id'])
assert split_to_groups['train'].isdisjoint(split_to_groups['val'])
assert split_to_groups['train'].isdisjoint(split_to_groups['test'])
assert split_to_groups['val'].isdisjoint(split_to_groups['test'])

split_augmented_image_counts = defaultdict(int)
for f in families:
    fid = f['family_id']
    split_name = split_map[fid]
    split_augmented_image_counts[split_name] += family_aug_counts[fid]

print('\nAugmented images per split (planned):')
for s in SPLITS:
    print(f'  {s}: {split_augmented_image_counts[s]}')


Class distribution AFTER augmentation (planned):
      class  count
   diabetes    500
nondiabetes    500

Augmented images per split (planned):
  train: 350
  val: 75
  test: 76


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [5]:
def safe_stem(name: str) -> str:
    # Keep filenames portable for Kaggle dataset publishing
    allowed = set('abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-_')
    return ''.join(ch if ch in allowed else '_' for ch in name)


# Prepare final output folders directly (no intermediate shared folder)
for split_name in SPLITS:
    split_dir = OUTPUT_BASE / split_name
    if split_dir.exists():
        shutil.rmtree(split_dir)
    split_dir.mkdir(parents=True, exist_ok=True)

for cls in class_to_originals.keys():
    for split_name in SPLITS:
        (OUTPUT_BASE / split_name / cls).mkdir(parents=True, exist_ok=True)


# Save originals + offline augmented images directly into split folders
final_counts = defaultdict(lambda: defaultdict(int))  # final_counts[split][class]
written_rows = []

for f in tqdm(families, desc='Saving augmented dataset'):
    fid = f['family_id']
    gid = f['group_id']
    cls = f['class_name']
    src = f['src_path']
    split_name = split_map[fid]

    img = cv2.imread(str(src))
    if img is None:
        print(f'[WARN] Could not read image: {src}')
        continue

    dst_dir = OUTPUT_BASE / split_name / cls

    # Save original
    orig_base = safe_stem(src.stem)
    orig_name = f'{fid}__orig__{orig_base}.jpg'
    orig_out = dst_dir / orig_name
    cv2.imwrite(str(orig_out), img, [int(cv2.IMWRITE_JPEG_QUALITY), int(CONFIG['save_jpg_quality'])])
    final_counts[split_name][cls] += 1
    written_rows.append({'filepath': str(orig_out), 'split': split_name, 'group_id': gid})

    # Save offline augmentations planned for this family
    n_aug = family_aug_counts[fid]
    for j in range(n_aug):
        aug_img = augment_once(img)
        aug_name = f'{fid}__aug_{j:03d}__{orig_base}.jpg'
        aug_out = dst_dir / aug_name
        cv2.imwrite(str(aug_out), aug_img, [int(cv2.IMWRITE_JPEG_QUALITY), int(CONFIG['save_jpg_quality'])])
        final_counts[split_name][cls] += 1
        written_rows.append({'filepath': str(aug_out), 'split': split_name, 'group_id': gid})


# Final report per class per split
rows = []
for split_name in SPLITS:
    for cls in sorted(class_to_originals.keys()):
        rows.append({
            'split': split_name,
            'class': cls,
            'count': final_counts[split_name][cls],
        })

final_df = pd.DataFrame(rows)
print('\nFinal image counts per class per split:')
print(final_df.to_string(index=False))

print('\nTotal images written:', int(final_df['count'].sum()))
print('Output root ready for Kaggle Dataset publish:', OUTPUT_BASE)

# Leakage checks on saved output
saved_df = pd.DataFrame(written_rows)
group_split_count = saved_df.groupby('group_id')['split'].nunique()
leaking_groups = group_split_count[group_split_count > 1]
print('Group leakage count:', int(leaking_groups.shape[0]))
assert leaking_groups.shape[0] == 0, 'Leakage detected: same group_id appears in multiple splits.'

# Optional exact-duplicate hash check across splits
if CONFIG['run_hash_leakage_check']:
    split_hashes = defaultdict(set)
    for row in written_rows:
        p = Path(row['filepath'])
        split_hashes[row['split']].add(md5_file(p))

    tv = len(split_hashes['train'].intersection(split_hashes['val']))
    tt = len(split_hashes['train'].intersection(split_hashes['test']))
    vt = len(split_hashes['val'].intersection(split_hashes['test']))

    print(f'Exact hash overlaps: train-val={tv}, train-test={tt}, val-test={vt}')
    assert tv == 0 and tt == 0 and vt == 0, 'Hash leakage detected across splits.'

# Build local zip files for easy download from /kaggle/working
for split_name in SPLITS:
    split_dir = OUTPUT_BASE / split_name
    zip_base = OUTPUT_BASE / split_name
    shutil.make_archive(str(zip_base), 'zip', root_dir=str(split_dir))

# Full bundle zip (train + val + test)
bundle_tmp = OUTPUT_BASE / 'tongue_preped_bundle'
if bundle_tmp.exists():
    shutil.rmtree(bundle_tmp)
bundle_tmp.mkdir(parents=True, exist_ok=True)
for split_name in SPLITS:
    shutil.copytree(OUTPUT_BASE / split_name, bundle_tmp / split_name)
shutil.make_archive(str(OUTPUT_BASE / 'tongue_preped_bundle'), 'zip', root_dir=str(bundle_tmp))
shutil.rmtree(bundle_tmp)

# Remove leftover metadata.csv from previous notebook versions if it exists
legacy_csv = OUTPUT_BASE / 'metadata.csv'
if legacy_csv.exists():
    legacy_csv.unlink()
    print('Removed old file:', legacy_csv)

print('\nDownloadable zip files created:')
print(' - /kaggle/working/train.zip')
print(' - /kaggle/working/val.zip')
print(' - /kaggle/working/test.zip')
print(' - /kaggle/working/tongue_preped_bundle.zip')

Saving augmented dataset:   0%|          | 0/499 [00:00<?, ?it/s]


Final image counts per class per split:
split       class  count
train    diabetes    349
train nondiabetes    350
  val    diabetes     74
  val nondiabetes     76
 test    diabetes     77
 test nondiabetes     74

Total images written: 1000
Output root ready for Kaggle Dataset publish: /kaggle/working
Metadata saved: /kaggle/working/metadata.csv
Group leakage count: 0
Exact hash overlaps: train-val=0, train-test=0, val-test=0


## Publish as Kaggle Dataset: `tongue_preped`

After running all cells:
1. Confirm the folders exist:
   - `/kaggle/working/train/<class_name>/`
   - `/kaggle/working/val/<class_name>/`
   - `/kaggle/working/test/<class_name>/`
2. Confirm downloadable zip files were created:
   - `/kaggle/working/train.zip`
   - `/kaggle/working/val.zip`
   - `/kaggle/working/test.zip`
   - `/kaggle/working/tongue_preped_bundle.zip`
3. Check leakage logs printed by the notebook:
   - `Group leakage count: 0`
   - `Exact hash overlaps: train-val=0, train-test=0, val-test=0`
4. In Kaggle Notebook right panel, click **Save Version**.
5. Choose **Save & Run All**.
6. In version options, enable **Create dataset from notebook output**.
7. Use title: `tongue_preped` (Kaggle slug will use `tongue-preped`).

This dataset can then be imported by your training notebook.

In [6]:
# Optional: create/update Kaggle Dataset from notebook output using Kaggle API
# Note: Kaggle dataset slug cannot contain underscore; use hyphen in slug.

import json
import subprocess

DATASET_TITLE = 'tongue_preped'
DATASET_SLUG = 'tongue-preped'
KAGGLE_USERNAME = os.environ.get('KAGGLE_USERNAME', 'edamhakim')
DATASET_ID = f'{KAGGLE_USERNAME}/{DATASET_SLUG}'

working_dir = Path('/kaggle/working')
publish_dir = working_dir / 'publish_dataset'

# Build clean publish folder so Kaggle upload contains only dataset files
if publish_dir.exists():
    shutil.rmtree(publish_dir)
publish_dir.mkdir(parents=True, exist_ok=True)

for split_name in SPLITS:
    shutil.copytree(working_dir / split_name, publish_dir / split_name)

metadata_path = publish_dir / 'dataset-metadata.json'
metadata = {
    'title': DATASET_TITLE,
    'id': DATASET_ID,
    'licenses': [{'name': 'CC0-1.0'}],
}

with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)

print('Wrote:', metadata_path)
print('Dataset id:', DATASET_ID)
print('Publishing from:', publish_dir)


def run_cmd(cmd):
    print('\n$', ' '.join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    out = (result.stdout or '') + (result.stderr or '')
    if out.strip():
        print(out)
    return result.returncode, out


create_rc, create_out = run_cmd(['kaggle', 'datasets', 'create', '-p', str(publish_dir), '--dir-mode', 'zip'])
create_failed = (create_rc != 0) or ('error' in create_out.lower())

if create_failed:
    print('Create failed or dataset already exists. Trying version update...')
    version_rc, version_out = run_cmd([
        'kaggle', 'datasets', 'version',
        '-p', str(publish_dir),
        '-m', 'Update prepared tongue dataset from notebook output',
        '--dir-mode', 'zip'
    ])
    version_failed = (version_rc != 0) or ('error' in version_out.lower())
    if version_failed:
        print('Could not publish via API. Use Kaggle UI: Save Version -> Create dataset from notebook output.')
    else:
        print('Dataset version uploaded successfully.')
else:
    print('Dataset created successfully.')

Wrote: /kaggle/working/dataset-metadata.json
Dataset id: edamhakim/tongue_preped

$ kaggle datasets create -p /kaggle/working --dir-mode zip
Starting upload for file test.zip
Upload successful: test.zip (21MB)
Starting upload for file metadata.csv
Upload successful: metadata.csv (249KB)
Starting upload for file train.zip
Upload successful: train.zip (79MB)
Starting upload for file state.db
Upload successful: state.db (2MB)
Starting upload for file .virtual_documents.zip
Upload successful: .virtual_documents.zip (5KB)
Starting upload for file val.zip
Upload successful: val.zip (10MB)
Dataset creation error: Slug can only contain alphanumeric or "-" characters

Dataset created successfully.
